In [ ]:
# ============================================================
# 06 — ABLATION: FROZEN vs ROLLING popularity (MIND)
# Q9 anti-gaming: popularity computed once (frozen, leaks+drifts) vs point-in-time (rolling).
# Fully self-contained MIND notebook. Hardcoded paths.
# ============================================================
!pip install lightgbm sentence-transformers -q
import os, glob, re, math, time, zipfile, numpy as np, pandas as pd, datetime as dt, lightgbm as lgb, random, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
random.seed(0)
# ---- hardcoded MIND paths (small: train -> dev for offline metrics) ----
TRAIN = "/kaggle/input/datasets/arashnic/mind-news-dataset/MINDsmall_train"
DEV   = "/kaggle/input/datasets/wrathofgod123/mind-dev/MINDsmall_dev"
SPLITS = [TRAIN, DEV]
NEWS = ["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH  = ["impression_id","user_id","time","history","impressions"]
_WORD = re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"

news = pd.concat([pd.read_csv(f"{d}/news.tsv", sep="\t", header=None, names=NEWS, quoting=3,
                 usecols=["news_id","category","title","abstract"]) for d in SPLITS]
                ).drop_duplicates("news_id").reset_index(drop=True)
news["title"] = news["title"].fillna(""); news["abstract"] = news["abstract"].fillna("")
cat_lut = {pfx(r.news_id):(r.category if isinstance(r.category,str) else "") for r in news.itertuples()}
ids = [pfx(r.news_id) for r in news.itertuples()]
corpus = [tok(f"{r.title} {r.abstract}") for r in news.itertuples()]
id_to_row = {x:i for i,x in enumerate(ids)}
title_lut = {pfx(r.news_id):tok(r.title) for r in news.itertuples()}
print("articles:", len(ids))


In [ ]:
from sentence_transformers import SentenceTransformer
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # English-specialised
emb_txt = [f"{r.title} {r.abstract}".strip() for r in news.itertuples()]
emb_mat = minilm.encode(emb_txt, batch_size=512, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=True)
emb_by_id = {ids[i]:emb_mat[i] for i in range(len(ids))}
print("MiniLM encoded:", emb_mat.shape)


In [ ]:
def load_beh(p):
    b = pd.read_csv(f"{p}/behaviors.tsv", sep="\t", header=None, names=BEH, quoting=3)
    b["t"] = pd.to_datetime(b["time"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"); return b

hist_lut = {}
first_seen = {}
click_ev = defaultdict(list); imp_ev = defaultdict(list)
_behs = [load_beh(p) for p in SPLITS]
for b in _behs:
    for u,h in zip(b["user_id"], b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)] = [pfx(x) for x in h.split()]
for b in _behs:
    for t,imps in zip(b["t"], b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tk in imps.split():
            nid = pfx(tk.split("-")[0])
            if nid not in first_seen or t < first_seen[nid]: first_seen[nid] = t
# click/impression events only from labeled splits (train+dev), not unlabeled test
for b in _behs[:2]:
    for t,imps in zip(b["t"], b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tk in imps.split():
            p = tk.split("-")
            if len(p)==2:
                nid = pfx(p[0]); imp_ev[nid].append(t)
                if p[1]=="1": click_ev[nid].append(t)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()

def hist_q(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];q=[]
    for x in ai: q.extend(title_lut.get(x,[]))
    return q
def hist_vecs(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];return [emb_by_id[x] for x in ai if x in emb_by_id]
def recency(aid,T,tau=6.0):
    fs=first_seen.get(aid)
    if fs is None or T is None: return 0.0
    dh=(T-fs).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def cnt(d,aid,T,w=None):
    tl=d.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T);return hi if w is None else hi-bisect_left(tl,T-dt.timedelta(hours=w))
def ctr(aid,T,w=None):
    c=cnt(click_ev,aid,T,w);s=cnt(imp_ev,aid,T,w);return c/s if s>0 else 0.0
def user_cats(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];cats=[cat_lut.get(x) for x in ai]
    tot=len([c for c in cats if c]);cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
def mm(x):
    lo,hi=x.min(),x.max();return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)
print("behavioural feature functions ready")

b_tr = load_beh(TRAIN); b_dv = load_beh(DEV)
print("train:",len(b_tr),"dev:",len(b_dv))
def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def mrr_i(s,lb):
    o=np.argsort(-s)
    for rank,idx in enumerate(o,1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def ndcg_i(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    ideal=np.sort(lb)[::-1][:k]
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(ideal))
    return float(dcg/idcg) if idcg>0 else 0.0
def iter_impressions(b_df):
    """yield (uid, T, candidates, labels) for impressions with a click + history."""
    for u,t,imps in zip(b_df["user_id"], b_df["t"], b_df["impressions"]):
        if not isinstance(imps,str) or pd.isna(t): continue
        cand=[];labs=[]
        for tk in imps.split():
            p=tk.split("-")
            if len(p)==2: cand.append(pfx(p[0])); labs.append(int(p[1]))
        if cand and sum(labs)>0:
            yield pfx(u), t, cand, np.array(labs)


In [ ]:
# frozen popularity: total clicks over TRAIN only, reused for every impression
from collections import Counter as _C
frozen_pop=_C()
for uid,T,cand,labs in iter_impressions(b_tr):
    for c,l in zip(cand,labs):
        if l==1: frozen_pop[c]+=1

def build_simple(b_df, mode):
    """minimal features to isolate the popularity effect: [pop, recency, ctr, slate]"""
    rows=list(iter_impressions(b_df)); X=[];y=[];g=[]
    for uid,T,cand,labs in rows:
        m=len(cand);F=[]
        for i,c in enumerate(cand):
            rec=recency(c,T)
            if mode=="rolling":
                p=cnt(click_ev,c,T,24); ct=ctr(c,T,24)
            else:
                p=frozen_pop.get(c,0); ct=0.0
            F.append([p,rec,ct,m])
        F=np.array(F,float); F[:,0]=mm(F[:,0]); F[:,1]=mm(F[:,1])
        X.append(F);y.append(labs);g.append(m)
    return np.vstack(X),np.concatenate(y),g

res={}
for mode in ("frozen","rolling"):
    Xtr,ytr,gtr=build_simple(b_tr,mode)
    Xdv,ydv,gdv=build_simple(b_dv,mode)
    rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=300,learning_rate=0.05,
                      num_leaves=31,min_child_samples=50,verbose=-1)
    rk.fit(Xtr,ytr,group=gtr)
    sc=rk.predict(Xdv);pos=0;aucs=[]
    for gi in gdv:
        a=auc_i(sc[pos:pos+gi],ydv[pos:pos+gi]);pos+=gi
        if a is not None: aucs.append(a)
    res[mode]=np.mean(aucs)
print("=== MIND frozen vs rolling popularity ===")
print(f"  frozen  AUC: {res['frozen']:.4f}")
print(f"  rolling AUC: {res['rolling']:.4f}")
print(f"  delta      : {res['rolling']-res['frozen']:+.4f}")
print("Rolling point-in-time popularity avoids leakage AND concept drift.")
